# BTC / ETH External Covariates Demo

This notebook inspects the normalized external covariates dataset produced by `scripts.download_external_covariates`.

Focus:
- load the partitioned parquet dataset from `cached_data/external_covariates`
- inspect the downloaded `btc_usd` and `eth_usd` series
- compute simple 5-minute returns and cross-series correlations
- visualize price paths, returns, and rolling correlation

In [1]:
# If needed in a fresh environment:
# %pip install -r ../requirements.txt

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "examples" else Path.cwd().resolve()
DATA_PATH = REPO_ROOT / "data" / "external_covariates"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Run scripts.download_external_covariates first."
    )


In [2]:
df = pd.read_parquet(DATA_PATH)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True, errors="coerce")
df = df.sort_values(["series_id", "timestamp_utc"], kind="stable").reset_index(drop=True)

crypto_df = df.loc[df["series_id"].isin(["btc_usd", "eth_usd"])].copy()
if crypto_df.empty:
    raise ValueError("No btc_usd / eth_usd rows found in external_covariates dataset.")

display(crypto_df.head())
display(
    crypto_df.groupby("series_id")
    .agg(
        rows=("value", "size"),
        start=("timestamp_utc", "min"),
        end=("timestamp_utc", "max"),
        min_close=("close", "min"),
        max_close=("close", "max"),
        mean_trade_count=("trade_count", "mean"),
        mean_volume=("volume", "mean"),
    )
    .reset_index()
)


In [3]:
wide = (
    crypto_df.pivot_table(index="timestamp_utc", columns="series_id", values="close", aggfunc="last")
    .sort_index()
)
returns = np.log(wide).diff().rename(columns=lambda c: f"{c}_log_return")
trade_count_wide = (
    crypto_df.pivot_table(index="timestamp_utc", columns="series_id", values="trade_count", aggfunc="last")
    .sort_index()
)

summary = pd.DataFrame(
    {
        "price_mean": wide.mean(),
        "price_std": wide.std(),
        "log_return_std": returns.std(),
        "trade_count_mean": trade_count_wide.mean(),
    }
)
display(summary)

corr = returns.corr()
display(corr)


In [4]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
wide.plot(ax=axes[0], linewidth=1.1)
axes[0].set_title("BTC and ETH close prices")
axes[0].set_ylabel("USD")

returns.plot(ax=axes[1], linewidth=0.8, alpha=0.8)
axes[1].set_title("5-minute log returns")
axes[1].set_ylabel("log return")

plt.tight_layout()
plt.show()


In [5]:
rolling_corr = returns["btc_usd_log_return"].rolling(288).corr(returns["eth_usd_log_return"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(returns["btc_usd_log_return"].dropna(), bins=80, ax=axes[0], color="#c57a00")
axes[0].set_title("BTC 5-minute return distribution")

sns.histplot(returns["eth_usd_log_return"].dropna(), bins=80, ax=axes[1], color="#3566b8")
axes[1].set_title("ETH 5-minute return distribution")

plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
rolling_corr.plot(color="#0b7d4b")
plt.title("Rolling 1-day correlation of BTC and ETH 5-minute log returns")
plt.ylabel("correlation")
plt.tight_layout()
plt.show()


Useful next steps:
- align these series to Polymarket benchmark timestamps with `benchmarks/covariate_utils.py`
- add lagged returns and realized-volatility features
- compare benchmark performance with and without external covariates